# Unit 08｜物理先验与真实闭环状态

## Goal

比较普通 GP、合理物理均值和错误物理均值，并将成本、可行性和人工批准加入 query。

本 Notebook 是确定性的人工教学实验，不是学习者已完成的研究，
也不是粘合剂实验结果。


## Setup

两种物理趋势都在生成隐藏标签前定义；教程只模拟审批状态，不连接真实设备。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

temperature_axis = np.linspace(30.0, 120.0, 13)
ratio_axis = np.linspace(0.1, 0.9, 11)
temperature, ratio = np.meshgrid(temperature_axis, ratio_axis)
X_all = np.column_stack([temperature.ravel(), ratio.ravel()])
candidate_ids = np.array([f"P{i:03d}" for i in range(len(X_all))])

def physics_mean(X):
    return (
        0.035 * X[:, 0]
        - 3.0 * (X[:, 1] - 0.60) ** 2
    )

def wrong_physics_mean(X):
    return (
        -0.020 * X[:, 0]
        + 2.5 * (X[:, 1] - 0.25) ** 2
    )

residual_truth = (
    0.55 * np.sin(X_all[:, 0] / 16.0)
    + 0.30 * np.cos(7.0 * X_all[:, 1])
)
oracle_values = physics_mean(X_all) + residual_truth

def offline_oracle(global_index):
    return float(oracle_values[int(global_index)])

feasible_mask = (
    (X_all[:, 0] <= 110.0)
    & (X_all[:, 1] >= 0.2)
    & (X_all[:, 1] <= 0.8)
)
experiment_cost = 1.0 + X_all[:, 0] / 120.0
feasible_indices = np.flatnonzero(feasible_mask)
initial_indices = feasible_indices[
    np.linspace(0, len(feasible_indices) - 1, 9, dtype=int)
]
initial_values = np.array([
    offline_oracle(index) for index in initial_indices
])
pool_indices = np.setdiff1d(np.arange(len(X_all)), initial_indices)
print("all/feasible/initial/pool:", len(X_all), len(feasible_indices), len(initial_indices), len(pool_indices))


## Steps

按顺序执行。每个变量第一次出现时，先确认它的类型、形状和标签权限。


### 1. 定义同规格 GP Pipeline

StandardScaler 只在已标注输入上 fit；固定核避免教程漂移。


In [ ]:
def make_gp():
    kernel = (
        ConstantKernel(1.0, constant_value_bounds="fixed")
        * RBF(1.0, length_scale_bounds="fixed")
        + WhiteKernel(0.02, noise_level_bounds="fixed")
    )
    return make_pipeline(
        StandardScaler(),
        GaussianProcessRegressor(
            kernel=kernel,
            optimizer=None,
            normalize_y=True,
        ),
    )


### 2. 普通、合理先验与错误先验 GP

两个结构化模型都先减去预先定义的趋势，再让同规格 GP 学残差。


In [ ]:
ordinary_gp = make_gp()
ordinary_gp.fit(
    X_all[initial_indices],
    initial_values,
)
ordinary_mean, ordinary_std = ordinary_gp.predict(
    X_all[pool_indices],
    return_std=True,
)

residual_gp = make_gp()
residual_y = (
    initial_values
    - physics_mean(X_all[initial_indices])
)
residual_gp.fit(X_all[initial_indices], residual_y)
residual_mean, structured_std = residual_gp.predict(
    X_all[pool_indices],
    return_std=True,
)
structured_mean = (
    physics_mean(X_all[pool_indices]) + residual_mean
)

wrong_residual_gp = make_gp()
wrong_residual_y = (
    initial_values
    - wrong_physics_mean(X_all[initial_indices])
)
wrong_residual_gp.fit(
    X_all[initial_indices],
    wrong_residual_y,
)
wrong_residual_mean, wrong_structured_std = (
    wrong_residual_gp.predict(
        X_all[pool_indices],
        return_std=True,
    )
)
wrong_structured_mean = (
    wrong_physics_mean(X_all[pool_indices])
    + wrong_residual_mean
)


### 3. 加入成本与可行性

不可行候选分数设为负无穷；成本必须为正。


In [ ]:
beta = 1.2
pool_cost = experiment_cost[pool_indices]
pool_feasible = feasible_mask[pool_indices]
structured_score = (
    structured_mean + beta * structured_std
) / pool_cost
structured_score = np.where(
    pool_feasible, structured_score, -np.inf
)
ordinary_score = (
    ordinary_mean + beta * ordinary_std
) / pool_cost
ordinary_score = np.where(
    pool_feasible, ordinary_score, -np.inf
)
wrong_structured_score = (
    wrong_structured_mean
    + beta * wrong_structured_std
) / pool_cost
wrong_structured_score = np.where(
    pool_feasible,
    wrong_structured_score,
    -np.inf,
)

def select_with_tie_break(score, ids):
    order = np.lexsort((ids.astype(str), -np.asarray(score)))
    return int(order[0])

structured_local = select_with_tie_break(
    structured_score,
    candidate_ids[pool_indices],
)
ordinary_local = select_with_tie_break(
    ordinary_score,
    candidate_ids[pool_indices],
)
wrong_structured_local = select_with_tie_break(
    wrong_structured_score,
    candidate_ids[pool_indices],
)
structured_global = int(pool_indices[structured_local])
ordinary_global = int(pool_indices[ordinary_local])
wrong_structured_global = int(
    pool_indices[wrong_structured_local]
)


### 4. 建立 proposed → approved → completed 状态

教学中模拟人工批准；真实系统必须由授权人员完成。


In [ ]:
proposal = pd.DataFrame([
    {
        "model": "ordinary_gp",
        "candidate_id": candidate_ids[ordinary_global],
        "temperature": X_all[ordinary_global, 0],
        "ratio": X_all[ordinary_global, 1],
        "cost": experiment_cost[ordinary_global],
        "feasible": feasible_mask[ordinary_global],
        "review_status": "proposed",
    },
    {
        "model": "physics_mean_plus_residual_gp",
        "candidate_id": candidate_ids[structured_global],
        "temperature": X_all[structured_global, 0],
        "ratio": X_all[structured_global, 1],
        "cost": experiment_cost[structured_global],
        "feasible": feasible_mask[structured_global],
        "review_status": "proposed",
    },
    {
        "model": "wrong_physics_mean_plus_residual_gp",
        "candidate_id": candidate_ids[
            wrong_structured_global
        ],
        "temperature": X_all[
            wrong_structured_global, 0
        ],
        "ratio": X_all[wrong_structured_global, 1],
        "cost": experiment_cost[wrong_structured_global],
        "feasible": feasible_mask[wrong_structured_global],
        "review_status": "proposed",
    },
])
print("算法只生成 proposed:")
print(proposal.round(3).to_string(index=False))

# 教学模拟：人工只批准满足可行性规则的候选。
reviewed = proposal.copy()
reviewed.loc[
    reviewed["feasible"], "review_status"
] = "approved"

# 标签揭示线：只有 approved 才模拟 Oracle 返回。
id_to_index = {
    candidate_id: index
    for index, candidate_id in enumerate(candidate_ids)
}
reviewed["observed_y"] = [
    offline_oracle(id_to_index[candidate_id])
    if status == "approved"
    else np.nan
    for candidate_id, status in zip(
        reviewed["candidate_id"],
        reviewed["review_status"],
    )
]
reviewed.loc[
    reviewed["observed_y"].notna(),
    "review_status",
] = "completed"
print("审核与离线 Oracle 返回后:")
print(reviewed.round(3).to_string(index=False))


### 5. 离线检查模型误差

该误差只用于教程诊断，不回头修改同一轮 query。


In [ ]:
offline_pool_truth = oracle_values[pool_indices]
ordinary_mae = float(np.mean(np.abs(
    offline_pool_truth - ordinary_mean
)))
structured_mae = float(np.mean(np.abs(
    offline_pool_truth - structured_mean
)))
wrong_structured_mae = float(np.mean(np.abs(
    offline_pool_truth - wrong_structured_mean
)))
ablation = pd.DataFrame([
    {"model": "ordinary_gp", "offline_pool_mae": ordinary_mae},
    {
        "model": "reasonable_prior_plus_residual_gp",
        "offline_pool_mae": structured_mae,
    },
    {
        "model": "wrong_prior_plus_residual_gp",
        "offline_pool_mae": wrong_structured_mae,
    },
]).sort_values("offline_pool_mae")
print(ablation.round(4).to_string(index=False))


## Checks

这些断言检查形状、预算和无重复等机械条件；通过断言不代表研究结论已经成立。


In [ ]:
assert np.all(experiment_cost > 0)
selected_globals = [
    ordinary_global,
    structured_global,
    wrong_structured_global,
]
assert feasible_mask[selected_globals].all()
assert proposal["review_status"].eq("proposed").all()
assert "observed_y" not in proposal.columns
assert reviewed["review_status"].eq("completed").all()
assert reviewed["observed_y"].notna().all()
assert structured_mean.shape == ordinary_mean.shape
assert wrong_structured_mean.shape == ordinary_mean.shape
assert set(ablation["model"]) == {
    "ordinary_gp",
    "reasonable_prior_plus_residual_gp",
    "wrong_prior_plus_residual_gp",
}
print("Unit 08 checks passed.")


## Next Steps

解释三组先验消融，并说明 PINN 还缺哪些模块才能进入主动学习。Unit 09 将组合完整研究包。
